# Day 64 — RAG Pipeline Clean
## Table-to-Natural-Language Fix + Full Ingestion Pipeline
### Problem solved: Revenue and net income queries broken due to poor table embedding
### Fix: Convert financial tables to prose before embedding

# Imports 

In [1]:
import re 
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from bs4 import BeautifulSoup

/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("apple_10k_2023.htm", 'r', encoding='utf-8' ,errors='ignore') as f:
    html  = f.read()

In [3]:
soup = BeautifulSoup(html,'html.parser')
for tag in soup(['script','style','meta','link']):
    tag.decompose()

# TableToNaturalLanguage

In [4]:
class TableToNaturalLanguage:
    def __init__(self,company):
        """
        Initializes the converter with specific document metadata.
        """
        self.company = company
        
    def is_financial_table(self,table_soup):
        """
        Analyzes a table to determine if it contains financial data 
        based on keyword presence and numerical patterns.
        """
        financial_keywords = ["net", "revenue", "income", "sales", "earnings", 
                          "profit", "loss", "assets", "cash", "cost"]
        numeric_pattern =re.compile(r'\$?[\d,]+\.?\d*')

        rows = table_soup.find_all("tr")

        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            first_cell_text = cells[0].get_text(strip=True).lower()
            has_keyword = any(kw in first_cell_text for kw in financial_keywords)

            has_numeric_value = False
            for cell in cells[1:]:
                cell_text = cell.get_text(strip=True)
                # We look for a cell that isn't empty and matches our financial number pattern
                if cell_text and numeric_pattern.search(cell_text):
                    has_numeric_value = True
                    break

            # If a single row contains both a keyword and a financial number, it's a winner
            if has_keyword and has_numeric_value:
                return True

        return False

    
    def convert(self, raw_html):
        soup = BeautifulSoup(raw_html, "html.parser")
        if not self.is_financial_table(soup):
            return []

        rows = soup.find_all("tr")
        col_map = {} # Temporary to find years and their order
        year_pattern = re.compile(r'20\d{2}')
        results = []

        # 1. Header Scan (Still needed to find WHICH years are in the table)
        for row in rows:
            cells = row.find_all(["td", "th"])
            for i, cell in enumerate(cells):
                text = cell.get_text(strip=True)
                if year_pattern.fullmatch(text):
                    col_map[i] = text
            if col_map: break

        if not col_map: return []
        
        # The One-Liner: Get years in the order they appear left-to-right
        years_in_order = [year for idx, year in sorted(col_map.items())]

        # 2. Data Extraction with Filtered Zip
        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2: continue
            
            metric = cells[0].get_text(strip=True)
            if not metric: continue

            numeric_values = []
            for cell in cells[1:]:
                val_text = cell.get_text(strip=True)
                # Filtering out (4), 11, $, etc. while keeping 162,560
                if re.search(r'\d{3,}', val_text):
                    numeric_values.append(val_text)
            
            
            for value, year in zip(numeric_values, years_in_order):
                sentence = f"The {metric} of {self.company} in the fiscal year {year} was {value}."
                results.append(sentence)
                
        return results

# Extraction from htmml 

In [5]:
def extract_sections_from_xbrl_html(soup, table_converter) -> list[dict]:
    """Semantic section extraction with font-weight detection and table parsing."""
    sections, section_num = [], 0
    current_title, current_text, current_length = "Preamble", [], 0
    all_elements = soup.find_all(['span', 'div', 'p', 'ix:nonnumeric', 'ix:nonfraction', 'table'])
    
    for elem in all_elements:
        if elem.name == 'table':
            table_prose = table_converter.convert(str(elem))
            if not table_prose: continue
            text, is_header = " ".join(table_prose), False
        else:
            if elem.find(['span', 'div']): 
                continue
            text = elem.get_text(strip=True)
            if len(text) < 5: 
                continue
            style = elem.get('style', '')
            is_header = ('font-weight:700' in style and len(text) < 150 and 
                         any(text.upper().startswith(k) for k in ['ITEM', 'PART', 'CONSOLIDATED', 'NOTE']))
        
        if is_header and current_text and current_length > 200:
            section_num += 1
            sections.append({
                "section_num": section_num, 
                "section_title": current_title, 
                "text": " ".join(current_text), 
                "char_count": current_length
            })
            current_title, current_text, current_length = text[:100], [], 0
        else:
            current_text.append(text)
            current_length += len(text)
            if current_length > 1500:
                section_num += 1
                sections.append({
                    "section_num": section_num, 
                    "section_title": current_title, 
                    "text": " ".join(current_text), 
                    "char_count": current_length
                })
                current_text, current_length = [], 0
    
    if current_text:
        sections.append({
            "section_num": section_num + 1, 
            "section_title": current_title, 
            "text": " ".join(current_text), 
            "char_count": current_length
        })
    return sections

In [6]:
def build_chunks_with_citations(sections, company, doc_type, fiscal_year, source_file):
    """Quality filtering and metadata assignment."""
    financial_signals = ["revenue", "income", "profit", "sales", "earnings", "cash", "assets", "net", "total"]
    chunks = []
    for section in sections:
        if 'http://' in section["text"] or 'https://' in section["text"]: 
            continue
        quality_score = sum(1 for w in financial_signals if w in section["text"].lower()) / len(financial_signals)
        if quality_score < 0.2: 
            continue
        chunks.append({
            "chunk_id": f"{company.lower().replace(' ', '_')}_{fiscal_year}_s{section['section_num']:03d}",
            "text": section["text"], 
            "company": company, 
            "doc_type": doc_type, 
            "fiscal_year": fiscal_year,
            "section_title": section["section_title"], 
            "source_file": source_file, 
            "section_num": section['section_num'],
            "quality_score": round(quality_score, 3)
        })
    return chunks

# enrich chunk text

In [7]:
def enrich_chunk_text(chunk):
    return f"{chunk['section_title']}:\n{chunk['text']}"

In [8]:
converter = TableToNaturalLanguage(company="Apple Inc.")
sections = extract_sections_from_xbrl_html(soup, converter)
chunks = build_chunks_with_citations(sections, "Apple Inc.", "10-K", "FY2023", "aapl-20230930.htm")
enriched_texts = [enrich_chunk_text(c) for c in chunks]

In [9]:
print(f"Sections extracted: {len(sections)}")
print(f"Chunks after quality filter: {len(chunks)}")

Sections extracted: 146
Chunks after quality filter: 74


In [10]:
for c in chunks:
    if "96,995" in c["text"]:
        print(f"ID: {c['chunk_id']}")
        print(f"Section: {c['section_title']}")
        print(f"Quality: {c['quality_score']}")
        print(f"Preview: {c['text'][:200]}")

ID: apple_inc._FY2023_s073
Section: CONSOLIDATED STATEMENTS OF OPERATIONS
Quality: 0.556
Preview: (In millions, except number of shares, which are reflected in thousands, and per-share amounts) Years ended September 30,2023 September 24,2022 September 25,2021 Net sales: Products 298,085 298,085 31
ID: apple_inc._FY2023_s074
Section: CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME
Quality: 0.333
Preview: (In millions) Years ended September 30,2023 September 24,2022 September 25,2021 Net income 96,995 96,995 99,803 99,803 94,680 94,680 Other comprehensive income/(loss): Change in foreign currency trans
ID: apple_inc._FY2023_s077
Section: CONSOLIDATED STATEMENTS OF SHAREHOLDERS’ EQUITY
Quality: 0.444
Preview: (In millions, except per-share amounts) Years ended September 30,2023 September 24,2022 September 25,2021 Total shareholders’ equity, beginning balances 50,672 50,672 63,090 63,090 65,339 65,339 Commo
ID: apple_inc._FY2023_s078
Section: CONSOLIDATED STATEMENTS OF CASH FLOWS
Quality: 

# Storing in chromadb

In [11]:
client = chromadb.PersistentClient(path="./chroma_db/")
try: 
    client.delete_collection("financial_docs_v2")
except: 
    pass
collection = client.create_collection(name="financial_docs_v2", metadata={"hnsw:space": "cosine"})

In [12]:
# Embedding
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(enriched_texts, batch_size=32, show_progress_bar=True)

Loading weights: 100%|█████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4773.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.08s/it]


In [13]:
collection.add(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=enriched_texts,
    metadatas=[{k: v for k, v in c.items() if k != 'text'} for c in chunks]
)

In [14]:
tokenized_enriched = [text.lower().split() for text in enriched_texts]
bm25 = BM25Okapi(tokenized_enriched)

In [15]:
print(chunks[0].keys())
print(chunks[0])

dict_keys(['chunk_id', 'text', 'company', 'doc_type', 'fiscal_year', 'section_title', 'source_file', 'section_num', 'quality_score'])
{'chunk_id': 'apple_inc._FY2023_s012', 'text': 'AppleCare The Company offers a portfolio of fee-based service and support products under the AppleCare brand. The offerings provide priority access to Apple technical support, access to the global Apple authorized service network for repair and replacement services, and in many cases additional coverage for instances of accidental damage or theft and loss, depending on the country and type of product. Cloud Services The Company’s cloud services store and keep customers’ content up-to-date and available across multiple Apple devices and Windows personal computers. Digital Content The Company operates various platforms, including the App Store that allow customers to discover and download applications and digital content, such as books, music, video, games and podcasts. The Company also offers digital content

In [16]:
print(f"Collection: {collection.name}")
print(f"Chunks stored: {collection.count()}")
print(f"BM25 vocab: {len(bm25.idf)} unique terms")

Collection: financial_docs_v2
Chunks stored: 74
BM25 vocab: 3588 unique terms


# retrieve with citation 

In [18]:
def retreive_with_citations(query: str, 
                             bm25_index, 
                             documents: list,
                             chunks: list,
                             collection,
                             n_results: int = 3,
                             alpha: float = 0.7) -> list[dict]:
    """
    We need citation because it will be easy for the use to verify whether the answer is genuine or not 
    """

    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max > bm25_min:
        bm25_normalized = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
    else:
        bm25_normalized = bm25_scores

    # Dense scores
    dense_results = collection.query(
        query_texts=[query],
        n_results=len(documents)
    )
    dense_distances = np.array(dense_results["distances"][0])
    dense_similarity = 1 - dense_distances
    dense_ids = dense_results["ids"][0]
    all_ids = [chunk["chunk_id"] for chunk in chunks]

    dense_aligned = np.zeros(len(documents))
    for i, doc_id in enumerate(all_ids):
        if doc_id in dense_ids:
            dense_idx = dense_ids.index(doc_id)
            dense_aligned[i] = dense_similarity[dense_idx]

    # Hybrid scores
    hybrid_scores = (alpha * dense_aligned) + ((1 - alpha) * bm25_normalized)
    top_indices = np.argsort(-hybrid_scores)[:n_results]

    results = []
    for idx in top_indices:
        chunk = chunks[idx]
        
        # Build citation string
        # WHY formatted citation: LLM pastes this directly into answer
        citation = (f"{chunk['company']} {chunk['doc_type']} "
                   f"{chunk['fiscal_year']} — {chunk['section_title']}")
        
        results.append({
            "text": chunk["text"],
            "citation": citation,
            "company": chunk["company"],
            "doc_type": chunk["doc_type"],
            "fiscal_year": chunk["fiscal_year"],
            "section_title": chunk["section_title"],
            "hybrid_score": round(float(hybrid_scores[idx]), 4),
            "chunk_id": chunk["chunk_id"]
        })
    
    return results

In [27]:
def expand_financial_query(query: str) -> str:
    """
    query expansion: bridges vocabulary gap between
    natural language questions and numeric financial tables
    """
    expansions = {
        "revenue": "revenue net sales total sales income",
        "profit": "profit net income earnings",
        "cash flow": "cash flow operations operating activities",
        "expenses": "expenses cost of sales operating expenses",
        "assets": "assets current assets total assets",
        "debt": "debt liabilities long-term debt",
        "earnings per share": "earnings per share EPS diluted basic",
        "net income": "net income profit earnings comprehensive income statements operations",
        "total revenue": "total revenue net sales total net sales",
    }
    
    expanded = query
    query_lower = query.lower()
    
    for term, expansion in expansions.items():
        if term in query_lower:
            expanded = f"{expanded} {expansion}"
    
    return expanded

# Test query expansion
test_queries_expanded = [
    "What is Apple's total revenue for 2023?",
    "What is Apple's net income?",
    "Apple cash flow from operations"
]

print("Query expansion results:")
for q in test_queries_expanded:
    expanded = expand_financial_query(q)
    print(f"\nOriginal: {q}")
    print(f"Expanded: {expanded}")

Query expansion results:

Original: What is Apple's total revenue for 2023?
Expanded: What is Apple's total revenue for 2023? revenue net sales total sales income total revenue net sales total net sales

Original: What is Apple's net income?
Expanded: What is Apple's net income? net income profit earnings comprehensive income statements operations

Original: Apple cash flow from operations
Expanded: Apple cash flow from operations cash flow operations operating activities


In [28]:
def citation_formatted_answer(query: str,
                             retrieved_results: list[dict]) -> str:
    context_parts = []
    
    for i, result in enumerate(retrieved_results):
        context_parts.append(
            f"[Source {i+1}: {result['citation']}]\n{result['text'][:500]}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are a financial analyst assistant. Answer the question using ONLY the provided sources below.
For every fact you state, cite the source number in brackets like [Source 1].
If the sources don't contain enough information, say so explicitly.

Sources:
{context}

Question: {query}

Answer:"""
    
    return prompt

In [29]:
for query in ["What is Apple's total revenue for 2023?", "What is Apple's net income?"]:
    expanded = expand_financial_query(query)
    results = retreive_with_citations(expanded, bm25, enriched_texts, chunks, collection)
    print(f"\nQuery: '{query}'")
    for r in results[:2]:
        print(f"  Score: {r['hybrid_score']} | Section: {r['section_title']}")
        print(f"  Preview: {r['text'][:150]}")


Query: 'What is Apple's total revenue for 2023?'
  Score: 0.7545 | Section: Note 2 –Revenue
  Preview: iPhone 200,583 200,583 205,489 205,489 191,973 191,973 29,357 29,357 40,177 40,177 35,190 35,190 28,300 28,300 29,292 29,292 31,862 31,862 Wearables, 
  Score: 0.6664 | Section: Item 6.    [Reserved]
  Preview: Change Change Research and development 29,915 26,251 21,914 Percentage of total net sales Selling, general and administrative 24,932 25,094 21,973 Per

Query: 'What is Apple's net income?'
  Score: 0.6647 | Section: Note 2 –Revenue
  Preview: Note 3 –Earnings Per Share Earnings Per Share The following table shows the computation of basic and diluted earnings per share for 2023, 2022 and 202
  Score: 0.6536 | Section: CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME
  Preview: (In millions) Years ended September 30,2023 September 24,2022 September 25,2021 Net income 96,995 96,995 99,803 99,803 94,680 94,680 Other comprehensi


In [30]:
for i, chunk in enumerate(chunks):
    if chunk['section_num'] == 74:
        print(f"Index: {i}")
        print(f"ID: {chunk['chunk_id']}")
        print(f"Section: {chunk['section_title']}")
        break

Index: 32
ID: apple_inc._FY2023_s074
Section: CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME


In [31]:
query = "What is Apple's net income?"
expanded = expand_financial_query(query)

tokenized_query = expanded.lower().split()
bm25_scores = bm25.get_scores(tokenized_query)
bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
bm25_normalized = (bm25_scores - bm25_min) / (bm25_max - bm25_min)

dense_results = collection.query(query_texts=[expanded], n_results=len(enriched_texts))
dense_distances = np.array(dense_results["distances"][0])
dense_similarity = 1 - dense_distances
dense_ids = dense_results["ids"][0]
all_ids = [c["chunk_id"] for c in chunks]

dense_aligned = np.zeros(len(enriched_texts))
for i, doc_id in enumerate(all_ids):
    if doc_id in dense_ids:
        dense_idx = dense_ids.index(doc_id)
        dense_aligned[i] = dense_similarity[dense_idx]

hybrid_scores = (0.7 * dense_aligned) + (0.3 * bm25_normalized)

print(f"s074 hybrid score: {hybrid_scores[32]:.4f}")
print(f"s074 bm25 score: {bm25_normalized[32]:.4f}")
print(f"s074 dense score: {dense_aligned[32]:.4f}")
print(f"\nTop 5 scores:")
top5 = np.argsort(-hybrid_scores)[:5]
for idx in top5:
    print(f"  {chunks[idx]['chunk_id']} | {chunks[idx]['section_title'][:50]} | {hybrid_scores[idx]:.4f}")

s074 hybrid score: 0.6536
s074 bm25 score: 1.0000
s074 dense score: 0.5051

Top 5 scores:
  apple_inc._FY2023_s086 | Note 2 –Revenue | 0.6647
  apple_inc._FY2023_s074 | CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME | 0.6536
  apple_inc._FY2023_s072 | Item 8.    Financial Statements and Supplementary  | 0.6287
  apple_inc._FY2023_s119 | Note 13 –Segment Information and Geographic Data | 0.5971
  apple_inc._FY2023_s098 | Note 6 –Consolidated Financial Statement Details | 0.5906
